# Loanliness: data pipeline, balancing, clustering and visualisation

Reimplementation of the preprocessing / k-means / PCA + t-SNE parts of the paper. Sections follow the task list:
1. joined feature table, 2. cleaning, 3. encoding and scaling, 4. split, 5. balancing, 6. polynomial features, 7. k-means + per-cluster LightGBM, 8. PCA / t-SNE.

## 1. Build the joined feature table

In [1]:
import gc, json, warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

DATA = Path("dataset")
OUT = Path("artifacts")
OUT.mkdir(exist_ok=True)

CONFIG = {
    "seed": 42,
    "test_size": 0.2,
    "missing_col_thresh": 0.30,   # drop columns with more than this fraction missing
    "impute_mode": "mean",        # "mean": fill with train means (keep all rows), "drop_rows": paper-style
    "cat_label_max_card": 2,      # categoricals with <= this many levels are label encoded, rest one-hot
}

In [2]:
def agg_table(df, key, prefix, drop_cols=()):
    # numeric -> mean/min/max/sum, categorical -> one-hot then mean (share) and sum (count)
    df = df.drop(columns=list(drop_cols), errors="ignore")
    cat_cols = [c for c in df.select_dtypes(exclude="number").columns if c != key]
    num = df.drop(columns=cat_cols)
    num_cols = [c for c in num.columns if c != key]
    parts = [num.groupby(key)[num_cols].agg(["mean", "min", "max", "sum"])]
    if cat_cols:
        dummies = pd.get_dummies(df[cat_cols], dtype="float32", dummy_na=False)
        dummies[key] = df[key].values
        parts.append(dummies.groupby(key).agg(["mean", "sum"]))
    out = pd.concat(parts, axis=1)
    out.columns = [f"{prefix}_{a}_{b}".replace(" ", "_") for a, b in out.columns]
    out[f"{prefix}_COUNT"] = df.groupby(key).size().astype("float32")
    return out.astype("float32")


def read(name, **kw):
    df = pd.read_csv(DATA / name, **kw)
    for c in df.select_dtypes("float64").columns:
        df[c] = df[c].astype("float32")
    return df

In [3]:
# bureau_balance -> per bureau record -> merged into bureau -> per client
bb = read("bureau_balance.csv")
bb_agg = agg_table(bb, "SK_ID_BUREAU", "bb")
del bb; gc.collect()

bureau = read("bureau.csv")
bureau = bureau.merge(bb_agg, left_on="SK_ID_BUREAU", right_index=True, how="left")
bureau_agg = agg_table(bureau, "SK_ID_CURR", "bureau", drop_cols=["SK_ID_BUREAU"])
del bureau, bb_agg; gc.collect()
bureau_agg.shape

(305811, 179)

In [4]:
# previous_application: 365243 is a sentinel in the DAYS_* columns here too
prev = read("previous_application.csv")
day_cols = [c for c in prev.columns if c.startswith("DAYS_")]
prev[day_cols] = prev[day_cols].replace(365243, np.nan)
prev_agg = agg_table(prev, "SK_ID_CURR", "prev", drop_cols=["SK_ID_PREV"])
del prev; gc.collect()

pos = read("POS_CASH_balance.csv")
pos_agg = agg_table(pos, "SK_ID_CURR", "pos", drop_cols=["SK_ID_PREV"])
pos_agg["pos_N_LOANS"] = pos.groupby("SK_ID_CURR")["SK_ID_PREV"].nunique().astype("float32")
del pos; gc.collect()

inst = read("installments_payments.csv")
inst["DAYS_PAST_DUE"] = inst["DAYS_ENTRY_PAYMENT"] - inst["DAYS_INSTALMENT"]   # >0 means paid late
inst["PAYMENT_DIFF"] = inst["AMT_INSTALMENT"] - inst["AMT_PAYMENT"]            # >0 means underpaid
inst_agg = agg_table(inst, "SK_ID_CURR", "inst", drop_cols=["SK_ID_PREV"])
del inst; gc.collect()

cc = read("credit_card_balance.csv")
cc_agg = agg_table(cc, "SK_ID_CURR", "cc", drop_cols=["SK_ID_PREV"])
del cc; gc.collect()
[t.shape for t in (prev_agg, pos_agg, inst_agg, cc_agg)]

[(338857, 363), (337252, 40), (339587, 33), (103558, 95)]

In [5]:
# left join everything onto application_train
app = read("application_train.csv")
app = app.rename(columns={c: f"app_{c}" for c in app.columns if c not in ("SK_ID_CURR", "TARGET")})
df = app
for agg in (bureau_agg, prev_agg, pos_agg, inst_agg, cc_agg):
    df = df.merge(agg, left_on="SK_ID_CURR", right_index=True, how="left")
del app, bureau_agg, prev_agg, pos_agg, inst_agg, cc_agg; gc.collect()

assert len(df) == 307511 and df["SK_ID_CURR"].is_unique
print(df.shape, "| default rate:", df["TARGET"].mean().round(4))
df.columns.to_series().str.split("_").str[0].value_counts()

(307511, 832) | default rate: 0.0807


prev      363
bureau    179
app       120
cc         95
pos        40
inst       33
SK          1
TARGET      1
Name: count, dtype: int64

## 2. Clean invalid and missing values and split

Only stateless fixes happen before the split. The split itself (stratified, fixed seed) is the contract with Person B (saved in section 4); it has to come before anything that learns from the data, so it sits here. The column-drop rule and the imputer are fitted on the train split only.

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split

# DAYS_EMPLOYED == 365243 is a placeholder (~18% of clients), not a real value
df["app_DAYS_EMPLOYED_ANOM"] = (df["app_DAYS_EMPLOYED"] == 365243).astype("float32")
df["app_DAYS_EMPLOYED"] = df["app_DAYS_EMPLOYED"].replace(365243, np.nan)

# a client with no bureau/card/... records has NaN everywhere for that source; the count columns are really 0
count_cols = [c for c in df.columns if c.endswith(("_COUNT", "_N_LOANS"))]
df[count_cols] = df[count_cols].fillna(0)

ids = df["SK_ID_CURR"]
y = df["TARGET"].astype("int8")
X = df.drop(columns=["SK_ID_CURR", "TARGET"])
for c in X.select_dtypes(exclude="number").columns:
    X[c] = X[c].astype(object)
del df; gc.collect()

# stratified split before any fitting
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=CONFIG["test_size"], stratify=y, random_state=CONFIG["seed"])
del X; gc.collect()
print(X_train.shape, X_test.shape, y_train.mean().round(4), y_test.mean().round(4))

(246008, 831) (61503, 831) 0.0807 0.0807


In [7]:
# drop columns with >30% missing (rate measured on train only)
miss = X_train.isna().mean()
drop_cols = miss[miss > CONFIG["missing_col_thresh"]].index.tolist()
print(f"dropping {len(drop_cols)} of {X_train.shape[1]} columns")
print(pd.Series(drop_cols).str.split("_").str[0].value_counts().to_dict())

keep_cols = [c for c in X_train.columns if c not in drop_cols]
X_train, X_test = X_train[keep_cols], X_test[keep_cols]

dropping 222 of 831 columns
{'cc': 94, 'bureau': 69, 'app': 50, 'prev': 9}


In [8]:
num_cols = X_train.select_dtypes("number").columns.tolist()
cat_cols = X_train.select_dtypes(exclude="number").columns.tolist()

if CONFIG["impute_mode"] == "drop_rows":
    # paper style: remove any row with a missing value (no learning involved, so both splits are treated alike)
    keep_tr, keep_te = ~X_train.isna().any(axis=1), ~X_test.isna().any(axis=1)
    X_train, y_train = X_train[keep_tr], y_train[keep_tr]
    X_test, y_test = X_test[keep_te], y_test[keep_te]

# numeric -> train mean, categorical -> explicit "missing" level; fitted on train only
imputer = ColumnTransformer([
    ("num", SimpleImputer(strategy="mean"), num_cols),
    ("cat", SimpleImputer(strategy="constant", fill_value="missing"), cat_cols),
], verbose_feature_names_out=False).set_output(transform="pandas")

X_train_imp = imputer.fit_transform(X_train)
X_test_imp = imputer.transform(X_test)
X_train_imp[num_cols] = X_train_imp[num_cols].astype("float32")
X_test_imp[num_cols] = X_test_imp[num_cols].astype("float32")
assert not X_train_imp.isna().any().any() and not X_test_imp.isna().any().any()
print(X_train_imp.shape, X_test_imp.shape)

(246008, 609) (61503, 609)


In [9]:
# row cost of the paper's drop-rows strategy on the same columns, for the write-up
n_complete = int((~pd.concat([X_train, X_test]).isna().any(axis=1)).sum()) if CONFIG["impute_mode"] == "mean" else len(X_train) + len(X_test)
print(f"rows kept with mean imputation: {len(y_train) + len(y_test):,}")
print(f"rows that would survive drop-rows: {n_complete:,} ({n_complete / len(y):.1%})")

rows kept with mean imputation: 307,511
rows that would survive drop-rows: 142,965 (46.5%)


## 3. Encode and scale

Binary / low-cardinality categoricals are label encoded, the rest one-hot encoded. Numeric columns are standardised. Everything is fitted on train.

In [10]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

card = X_train_imp[cat_cols].nunique()
label_cols = card[card <= CONFIG["cat_label_max_card"]].index.tolist()
onehot_cols = card[card > CONFIG["cat_label_max_card"]].index.tolist()
print(len(label_cols), "label encoded,", len(onehot_cols), "one-hot encoded")

encoder = ColumnTransformer([
    ("label", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1, dtype="float32"), label_cols),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype="float32"), onehot_cols),
    ("num", "passthrough", num_cols),
], verbose_feature_names_out=False).set_output(transform="pandas")

X_train_enc = encoder.fit_transform(X_train_imp)
X_test_enc = encoder.transform(X_test_imp)

# scale only the original numeric columns (label/one-hot outputs stay 0/1)
scaler = ColumnTransformer([("num", StandardScaler(), num_cols)],
                           remainder="passthrough", verbose_feature_names_out=False).set_output(transform="pandas")
X_train_s = scaler.fit_transform(X_train_enc).astype("float32")
X_test_s = scaler.transform(X_test_enc).astype("float32")
feature_names = X_train_s.columns.tolist()
del X_train_enc, X_test_enc; gc.collect()
print(X_train_s.shape, X_test_s.shape)

3 label encoded, 8 one-hot encoded


(246008, 702) (61503, 702)


In [11]:
# everything needed to push new raw data (same columns as the joined table) through the same pipeline
joblib.dump({"keep_cols": keep_cols, "imputer": imputer, "encoder": encoder, "scaler": scaler,
             "feature_names": feature_names, "config": CONFIG}, OUT / "preprocessor.joblib")

def transform_new(raw: pd.DataFrame) -> pd.DataFrame:
    # raw: joined features after the DAYS_EMPLOYED fix and count-fill above, without ID/TARGET columns
    art = joblib.load(OUT / "preprocessor.joblib")
    x = raw[art["keep_cols"]]
    return art["scaler"].transform(art["encoder"].transform(art["imputer"].transform(x)))[art["feature_names"]]

# sanity check: reloaded pipeline reproduces the test matrix
assert np.allclose(transform_new(X_test.iloc[:200]).to_numpy(), X_test_s.iloc[:200].to_numpy(), atol=1e-4)

## 4. Save the splits and data dictionary

The split was made in section 2 (stratified, `seed` from `CONFIG`) before any fitting. Both splits are saved with `SK_ID_CURR` and `TARGET` in front of the feature columns, in a fixed column order recorded in `schema.json`.

In [12]:
def with_ids(Xs, ys):
    out = Xs.copy()
    out.insert(0, "TARGET", ys.to_numpy())
    out.insert(0, "SK_ID_CURR", ids.loc[Xs.index].to_numpy())
    return out.reset_index(drop=True)

train_out, test_out = with_ids(X_train_s, y_train), with_ids(X_test_s, y_test)
train_out.to_parquet(OUT / "train.parquet")
test_out.to_parquet(OUT / "test.parquet")
json.dump({"id": "SK_ID_CURR", "target": "TARGET", "features": feature_names, "config": CONFIG},
          open(OUT / "schema.json", "w"), indent=1)
print(train_out.shape, test_out.shape)

(246008, 704) (61503, 704)


In [13]:
# data dictionary: feature -> source file, encoding, and the original description where one exists
src_map = {"app": "application_train", "bureau": "bureau", "prev": "previous_application",
           "pos": "POS_CASH_balance", "inst": "installments_payments", "cc": "credit_card_balance"}
desc = pd.read_csv(DATA / "HomeCredit_columns_description.csv", encoding="latin-1")
desc = desc[desc["Table"].str.startswith("application")].set_index("Row")["Description"].to_dict()

def describe(f):
    prefix = f.split("_")[0]
    if f.startswith("bureau_bb_"):
        src = "bureau_balance (via bureau)"
    else:
        src = src_map.get(prefix, "derived")
    kind = ("label" if any(f == c for c in label_cols) else
            "one-hot" if any(f.startswith(c + "_") for c in onehot_cols) else "numeric")
    base = f[4:] if prefix == "app" else ""
    d = desc.get(base) or desc.get(base.rsplit("_", 1)[0], "") if base else ""
    if f == "app_DAYS_EMPLOYED_ANOM": d, src = "1 if DAYS_EMPLOYED was the 365243 placeholder", "application_train (derived)"
    return src, kind, d

dictionary = pd.DataFrame([(f, *describe(f)) for f in feature_names], columns=["feature", "source", "encoding", "description"])
dictionary.to_csv(OUT / "data_dictionary.csv", index=False)
dictionary.groupby(["source", "encoding"]).size()

source                       encoding
POS_CASH_balance             numeric      40
application_train            label         3
                             numeric      59
                             one-hot     101
application_train (derived)  numeric       1
bureau                       numeric      89
bureau_balance (via bureau)  numeric      21
credit_card_balance          numeric       1
installments_payments        numeric      33
previous_application         numeric     354
dtype: int64